# Evolutionary Strategies

<a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" align="left" src="https://i.creativecommons.org/l/by-sa/4.0/80x15.png" /></a>&nbsp;| Dennis G. Wilson | <a href="https://supaerodatascience.github.io/stochastic/">https://supaerodatascience.github.io/stochastic/</a>

## Overview

1. [$(1 + \lambda)$ ES](#oneplus)
2. [Approximating the gradient](#gradient)
3. [Canonical ES](#canonical)
4. [CMA-ES part 1: maximum likelihood estimation](#cmaes)
5. [CMA-ES part 2: step size adaptation](#step)

# <a name="oneplus">1.</a> The $(1 + \lambda)$ ES

La stratégie d'évolution la plus simple est la $(1+1)$. Dans cet algorithme, le meilleur individu généré est remplacé par un nouvel individu lorsque sa **valeur de fitness** est supérieure à celle du meilleur individu actuel

    Initialize x randomly in ℝ
    while not terminate
        x' = x + 𝑁(0, 1)
        if f(x′) < f(x)
            x = x'
    return x

Dans cet algorithme, à chaque étape, nous effectuons un tirage aléatoire selon une loi normale pour obtenir un nouveau point à proximité du point actuel. Toutefois, pour mieux orienter le déplacement depuis ce point initial, nous pourrions échantillonner plusieurs points au lieu d'un seul. Nous examinerons d'abord des exemples de fonctions continues, car leur espace de recherche est plus facile à visualiser.

In [ ]:
def himmelblau(x, y):
    return (x**2 + y - 11)**2 + (x + y**2 - 7)**2

def himmelblau_v(x):
    return himmelblau(x[0], x[1])

In [ ]:
def rosenbrock(x, y, a=1, B=100):
      return (a-x)**2 + B*((y-x**2))**2

def rosenbrock_v(x):
      return rosenbrock(x[0], x[1])

In [ ]:
def warped_rastrigin(x, y):
    x = (x + 2.7) / 3
    y = (y - 4.1) / 2
    return 20 + x**2 - 10 * np.cos(2*np.pi*x) + y**2 - 10 * np.cos(2*np.pi*y)

def warped_rastrigin_v(x):
    return warped_rastrigin(x[0], x[1])

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib import ticker, cm
import matplotlib.colors as colors
import numpy as np
import math

In [ ]:
X = np.arange(-5, 5, 0.1)
Y = np.arange(-5, 5, 0.1)
X, Y = np.meshgrid(X, Y)
Z = himmelblau(X, Y)

In [ ]:
fig = plt.figure(figsize=(8, 6))
cs = plt.contour(X, Y, Z, levels=100, cmap='Spectral',
                 norm=colors.Normalize(vmin=Z.min(), vmax=Z.max()), alpha=0.4)
fig.colorbar(cs)
plt.show()

Nous commencerons par un point $x$ choisi au hasard. Nous utiliserons ensuite une loi normale pour tirer 20 points autour de $x$.

In [ ]:
x = np.random.rand(2) * 4 - 2
print("position: ", x, "fitness :", himmelblau(x[0], x[1]))
x_t = np.array([x + np.random.normal(size=(2,)) for i in range(20)])
fig = plt.figure(figsize=(8, 6))
cs = plt.contour(X, Y, Z, levels=100, cmap='Spectral', norm=colors.Normalize(vmin=Z.min(), vmax=Z.max()), alpha=0.4)
plt.scatter(x_t[:, 0], x_t[:, 1], c='b')
plt.scatter(x[0], x[1], c='r')
fig.colorbar(cs);

Disposer de cet ensemble de points nous permettra d'explorer l'espace autour de $x$ avant de passer à l'étape suivante, ce qui peut mieux orienter le déplacement dans l'espace de recherche. Toutefois, ces points risquent de se chevaucher ou de couvrir des zones déjà explorées. La génération d'un trop grand nombre de points pourrait ralentir la recherche en nécessitant davantage d'évaluations de la fonction objectif, une opération souvent coûteuse dans de nombreuses applications.

La réalisation d'une optimisation aléatoire impliquant l'échantillonnage de plus d'un point conduit à la stratégie d'évolution $(1+\lambda)$.  

    Initialize x randomly in ℝ
    while not terminate
        x_p = x
        for i in [1,λ]
            x_i = x_p + 𝑁(0, 1)
            if f(x_i) < f(x)
                x = x_i
        x_p = x
    return x_p

Dans le cadre des stratégies évolutionnaires, on désigne $x$ comme un parent et tous les points $x'$ comme des descendants.  
  
Chaque point peut également être qualifié d'individu au sein d'une population, et $f(x)$ est appelé la valeur de fitness de l'individu. Chaque itération de l'algorithme est appelée une génération.

<div class="alert alert-info">

La notation $(\mu/\rho, \lambda)$ ou $(\mu/\rho + \lambda)$ désigne la configuration des parents et de la descendance. Dans cette notation, $\mu$ représente le nombre de parents, $\rho$ le nombre de parents participant à la création de la descendance, et $\lambda$ le nombre de descendants. La notation $(\mu/\rho + \lambda)$ signifie que les parents peuvent être inclus dans la population suivante, tandis que $(\mu/\rho, \lambda)$ signifie qu'ils ne le sont pas. Ainsi, $(1 + \lambda)$ indique qu'un parent participe à la création d'une descendance de $\lambda$ individus et peut être inclus dans la génération suivante (c'est-à-dire que si $f(x) < f(x')$ pour tout $x'$, $x$ reste inchangé).

</div>

In [ ]:
def oneplus_lambda(x, f, gens=200, lam=20):
    x_best = x
    f_best = f(x)
    fits = np.zeros(gens)
    for g in range(gens):
        N = np.random.normal(size=(lam, len(x)))
        for i in range(lam):
            ind = x + N[i, :]
            fi = f(ind)
            if fi < f_best:
                f_best = fi
                x_best = ind
        x = x_best
        fits[g] = f_best
    return fits, x_best

Exécutons la stratégie d'évolution $(1+\lambda)$ sur la fonction de Himmelblau et enregistrons l'évolution au fil de plusieurs essais.

In [ ]:
f = himmelblau_v
runs = []
for i in range(5):
    x = np.random.rand(2)*4-2
    fits, _ = oneplus_lambda(x, f)
    runs += [fits]
runs = np.array(runs)
runs_mean = np.mean(runs, 0)
runs_std = np.std(runs, 0)

In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.fill_between(range(len(runs[0])), runs_mean+0.5*runs_std, runs_mean-0.5*runs_std, alpha=0.5)
plt.plot(range(len(runs[0])), runs_mean)
plt.yscale('log')

<div class="alert alert-success">
    <h3>Exercice 1 </h3>
    
Étudiez l'impact du paramètre $\lambda$ en le modifiant et en relançant l'optimisation.  
  
Visualisez les deux autres fonctions : Rosenbrock et Rastrigin. Testez l'algorithme $(1+\lambda)$ sur ces autres fonctions.
    
</div>

# <a name="gradient">2.</a> Approximation du gradient

Dans la stratégie d'évolution $(1+\lambda)$, on passe du meilleur point d'une population au meilleur point de la population suivante, échantillonnée aléatoirement.  
  
Toutefois, en exploitant les informations de la population, il est possible de faire mieux : on peut se déplacer dans la direction du gradient estimé à partir des valeurs de fitness de la population.  
  
Cette approche permet également de mieux tirer parti de l'ensemble des informations de la population, en combinant toutes les données de fitness au lieu de se limiter à la meilleure valeur.

In [ ]:
x = np.random.rand(2) * 4 - 2
x_t = np.array([x + np.random.normal(size=(2,)) for i in range(10)])
fits = himmelblau(x_t[:, 0], x_t[:, 1])

fig = plt.figure(figsize=(8, 6))
norm = colors.Normalize(vmin=Z.min(), vmax=Z.max())
cs = plt.contour(X, Y, Z, levels=100, cmap='Spectral', alpha=0.4, norm=norm)
plt.quiver(x_t[:, 0], x_t[:, 1], x_t[:, 0]-x[0], x_t[:, 1]-x[1],
           fits, scale=fits/100, scale_units='xy', cmap='Spectral', norm=norm)
plt.scatter(x[0], x[1], c='k')
fig.colorbar(cs)
plt.show()

Tout d'abord, nous normalisons toutes les valeurs de fitness à l'aide de la moyenne $\mu(f(x))$ et de l'écart-type $\sigma(f(x))$ de la population. L'objectif est de fonder le déplacement sur le fitness relatif au sein de la population, plutôt que sur le fitness absolu dans l'espace de recherche.  

$A = \frac{f(x) - \mu(f(x))}{\sigma(f(x))}$

Définissons ensuite un vecteur en chaque point, $x_i - x$. Notons que, puisque $x_i = x + N(0, 1)$, nous pourrions simplement conserver le tirage aléatoire selon la loi $N(0, 1)$ comme vecteur.  

$N_i = x_i - x$

Nous multiplions $A$ par tous les vecteurs individuels, ce qui met à l'échelle la norme de chaque vecteur en fonction de la valeur du fitness relatif. Enfin, nous sommons tous les vecteurs et divisons par $\lambda$ pour obtenir la moyenne pondérée.

$\frac{1}{\lambda}\sum_i A_i N_i$

Cela équivaut au produit scalaire du vecteur de fitness $A$ et du vecteur de position $N$. Nous considérons cela comme une approximation du gradient de $f$, car il indique la variation de $f$ lors d'un déplacement selon $x$. Nous pouvons descendre le long de cette approximation du gradient pour minimiser:  

$\nabla f \approx -\frac{A \cdot N}{\lambda}$

In [ ]:
A = (fits - np.mean(fits)) / np.std(fits)
N = x_t - x
G = -np.dot(A, N) / 10

In [ ]:
fig = plt.figure(figsize=(8, 6))
norm = colors.Normalize(vmin=Z.min(), vmax=Z.max())
cs = plt.contour(X, Y, Z, levels=100, cmap='Spectral', alpha=0.4, norm=norm)
plt.quiver(x_t[:, 0], x_t[:, 1], x_t[:, 0]-x[0], x_t[:, 1]-x[1],
           fits, scale=fits/100, scale_units='xy', cmap='Spectral', norm=norm)
plt.quiver(x[0], x[1], G[0], G[1])
plt.scatter(x[0], x[1], c='k')
fig.colorbar(cs)
plt.show()

Nous disposons de la direction approximative du gradient, mais quelle distance devons-nous parcourir dans cette direction?  
  
Pour l'instant, nous définirons une variable de taux d'apprentissage $\alpha$ représentant l'amplitude fixe du déplacement; la mise à jour de $x$ s'écrit alors

$x = x - \alpha\frac{A \cdot N}{\lambda}$

Rassemblons ces éléments pour former une stratégie d'évolution. Il s'agit d'une stratégie d'évolution de type $(\mu, \lambda)$, car plusieurs individus ($\mu$) contribuent à la génération suivante ; toutefois, ces individus ne sont pas conservés pour cette génération.  

    Initialize x randomly in ℝ
    while not terminate
        for i in [1,λ]
            N_i = 𝑁(0, 1)
            F_i = f(x + N_i)
        A = (F−𝜇(F))/𝜎(F)
        x = x - 𝛼(A⋅N)/𝜆

In [ ]:
def mu_lambda(x, fitness, gens=200, lam=20, alpha=0.20, verbose=False):
    x_best = x
    f_best = fitness(x)
    fits = np.zeros(gens)
    for g in range(gens):
        N = np.random.normal(size=(lam, len(x)))
        F = np.zeros(lam)
        for i in range(lam):
            ind = x + N[i, :]
            F[i] = fitness(ind)
            if F[i] < f_best:
                f_best = F[i]
                x_best = ind
                if verbose:
                    print(g, " ", f_best)
        fits[g] = f_best
        mu_f = np.mean(F)
        std_f = np.std(F)
        A = F
        if std_f != 0:
            A = (F - mu_f) / std_f
        x = x - alpha * np.dot(A, N) / lam
    return fits, x_best

In [ ]:
f = himmelblau_v
runs = []
for i in range(5):
    x = np.random.randn(2)
    fits, _ = mu_lambda(x, f)
    runs += [fits]
runs = np.array(runs)
runs_mean = np.mean(runs, 0)
runs_std = np.std(runs, 0)

In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.fill_between(range(len(runs[0])), runs_mean+0.5*runs_std, runs_mean-0.5*runs_std, alpha=0.5)
plt.plot(range(len(runs[0])), runs_mean)
plt.yscale('log')

<div class="alert alert-success">
    <h3>Exercice 2</h3>
    
Étudiez l'impact du paramètre $\alpha$ en le modifiant et en relançant l'optimisation. Comparez les meilleurs paramètres obtenus avec les résultats de la stratégie d'évolution $(1+\lambda)$.
    
</div>

# <a name="canonical">3.</a> Canonical ES

Nous pouvons formuler deux observations concernant notre stratégie d'évolution $(\mu, \lambda)$ actuelle :

1. Tous les points $\lambda$ ne sont pas nécessairement utiles pour la mise à jour du centroïde.
2. L'information exacte sur le gradient peut s'avérer moins utile que le *rang* relatif du fitness des individus.

Illustrons ce point à l'aide de la fonction de Himmelblau. Au lieu d'utiliser tous les points $\lambda$, nous utiliserons une valeur de $\mu$ plus faible, par exemple $\frac{1}{2}\lambda$. Nous créerons également une liste de poids décroissant de manière exponentielle, du premier au dernier.

In [ ]:
mu = 5
w = np.log(mu + 1/2) - np.log(np.arange(1, mu+1))
w /= np.sum(w)
w

Comme précédemment, nous créons un point de départ aléatoire, puis nous ajoutons un bruit gaussien pour générer une descendance.

In [ ]:
np.random.seed(123)
x = np.random.rand(2) * 4 - 2
N = np.random.normal(size=(10,2))
x_t = x + N
fits = himmelblau(x_t[:, 0], x_t[:, 1])
fits

Au lieu d'utiliser directement les valeurs de fitness, nous allons ordonner et sélectionner les meilleurs individus et effectuer la mise à jour en utilisant leurs valeurs pondérées.

In [ ]:
sorted_ids = np.argsort(fits)
weighted = N[sorted_ids[:mu]] * w.reshape(mu, 1)
weighted

In [ ]:
fig = plt.figure(figsize=(8, 6))
norm = colors.Normalize(vmin=Z.min(), vmax=Z.max())
cs = plt.contour(X, Y, Z, levels=100, cmap='Spectral', alpha=0.4, norm=norm)
plt.quiver(x_t[:, 0], x_t[:, 1], x_t[:, 0]-x[0], x_t[:, 1]-x[1],
           fits, scale=fits/100, scale_units='xy', cmap='Spectral', norm=norm, alpha=0.4)
plt.quiver(x[0]+weighted[:, 0], x[1]+weighted[:, 1], weighted[:, 0], weighted[:, 1],
           w, scale_units='xy', cmap='Spectral', norm=norm)
plt.scatter(x[0], x[1], c='k')
fig.colorbar(cs)
plt.show()

Considérez la fonction de Rosenbrock. Comment pensez-vous que les classements basés sur le gradient se comparent aux informations fournies par le gradient lui-même ? Seront-ils très similaires ou différents ? Tracez un échantillon pour l'examiner.

Le premier point est clair : l'utilisation de tous les points lors de la mise à jour du centroïde pourrait l'éloigner d'optima voisins.  
  
Le second point est un peu plus délicat à saisir, mais l'idée sous-jacente est la suivante: toute transformation de l'espace de recherche qui **préserve le classement** des $\mu$ meilleurs individus ne modifie pas la mise à jour. Cette propriété d'invariance favorise la recherche en renforçant le pouvoir prédictif de la fonction de fitness, grâce à la création de classes d'équivalence de problèmes.  

Hansen, N., Ros, R., Mauny, N., Schoenauer, M., & Auger, A. (2011). Impacts of invariance in search: When CMA-ES and PSO face ill-conditioned and non-separable problems. Applied Soft Computing, 11(8), 5755-5769.

L'intégration de ces deux modifications à la stratégie d'évolution $(\mu, \lambda)$ précédente aboutit à une stratégie d'évolution appelée **stratégie d'évolution canonique**.  

<img src="https://github.com/SupaeroDataScience/stochastic/blob/master/notebooks/imgs/canonical.png?raw=1" width="40%" height="auto">

Chrabaszcz, P., Loshchilov, I., & Hutter, F. (2018, July). Back to basics: benchmarking canonical evolution strategies for playing Atari. In Proceedings of the 27th International Joint Conference on Artificial Intelligence (pp. 1419-1426). [code](https://github.com/PatrykChrabaszcz/Canonical_ES_Atari)

Bien que cette stratégie évolutionnaire soit très simple, elle a été utilisée pour faire évoluer des réseaux de neurones profonds capables de jouer à des jeux Atari. Elle rivalise avec une stratégie évolutionnaire développée par OpenAI, et toutes deux sont compétitives face à l'apprentissage par renforcement profond.

<img src="https://github.com/SupaeroDataScience/stochastic/blob/master/notebooks/imgs/canonical_results.png?raw=1" width="80%" height="auto">

<div class="alert alert-warning">

<h3>Stratégies aisement parallélisable</h3>
    
Tous les exemples de stratégies d'évolution (ES) examinés jusqu'ici reposent sur une évaluation séquentielle des individus au sein d'un thread unique. Toutefois, les évaluations peuvent être effectuées entièrement en parallèle.   
      
L'évaluation constitue, de loin, l'étape la plus coûteuse en ressources de calcul; sa nature « aisément parallélisable » implique que la durée d'exécution réelle des algorithmes évolutionnaires peut être considérablement réduite grâce à une parallélisation sur plusieurs threads.

</div>

In [ ]:
def canonical(x, fitness, gens=200, mu=10, lam=20, verbose=False):
    # best values and fitness vector for logging
    x_best = x
    f_best = fitness(x)
    fits = np.zeros(gens)

    # calculate weight list
    w = np.log(mu + 1/2) - np.log(np.arange(1, mu+1))
    w /= np.sum(w)

    # generation loop
    for g in range(gens):
        # calculate new individuals
        N = np.random.normal(size=(lam,2))
        x_t = x + N
        # evaluate individuals
        F = [fitness(x_t[i, :]) for i in range(lam)]
        # store best individual for logging
        for i in range(lam):
            if F[i] < f_best:
                f_best = F[i]
                x_best = x_t[i]
                if verbose:
                    print(g, " ", f_best)
        fits[g] = f_best
        # use weighted sum to update center
        sorted_ids = np.argsort(F)
        weighted = N[sorted_ids[:mu]] * w.reshape(mu, 1)
        s = np.mean(weighted, axis=0)
        x = x + s

    return fits, x_best

<div class="alert alert-success">
    <h3>Exercice 3</h3>  
    
Comparez les stratégies d'évolution $(1+\lambda)$, $(\mu,\lambda)$ et canonique sur les fonctions de Rosenbrock, Himmelblau et Rastrigin. L'une de ces trois stratégies est-elle nettement supérieure aux autres ?
</div>

In [ ]:
# Voir ES_Solution1.ipynb

In [ ]:
compare_es([oneplus_lambda, mu_lambda, canonical], himmelblau_v)

# <a name="cmaes">4.</a> CMAES part 1: Maximum Likelihood Estimation

La stratégie d'évolution avec adaptation de la matrice de covariance, ou CMA-ES [1, 2], est l'un des algorithmes évolutionnaires les plus connus en général et constitue un algorithme de pointe pour l'optimisation continue. La force de cette méthode réside dans sa capacité à adapter la distribution utilisée pour générer la population suivante en fonction de la distribution actuelle des individus. Jusqu'à présent, nous étions limités à une distribution normale avec un paramètre $\sigma$ statique. Grâce à sa distribution adaptative, le CMA-ES permet de parcourir plus rapidement les espaces de recherche et de converger avec plus de précision vers les points optimaux.  

[1] Hansen, Nikolaus, and Andreas Ostermeier. "Adapting arbitrary normal mutation distributions in evolution strategies: The covariance matrix adaptation." Proceedings of IEEE international conference on evolutionary computation. IEEE, 1996.

[2] Hansen, Nikolaus, and Andreas Ostermeier. "Completely derandomized self-adaptation in evolution strategies." Evolutionary computation 9.2 (2001): 159-195.

Plus précisément, les améliorations apportées par CMA-ES par rapport aux stratégies d'évolution précédentes que nous avons étudiées sont les suivantes :  

+ il combine des informations provenant de plusieurs individus (comme dans les méthodes $(\mu, \lambda)$ et les stratégies d'évolution canoniques) ;
+ il combine des informations issues de plusieurs générations ;
+ il transforme la distribution de la nouvelle population pour l'adapter à l'espace de recherche ;
+ il adapte la taille du pas pour éviter une convergence prématurée.

Une amélioration propre à CMA-ES, que nous avons déjà évoquée, consiste à utiliser le **rang de fitness** (ou rang de performance) plutôt que la valeur de fitness brute pour effectuer la mise à jour. Toutefois, au lieu de mettre à jour directement un centroïde, CMA-ES met à jour une matrice de covariance.

Pour y parvenir, CMA-ES s'appuie sur deux principes : l'estimation du maximum de vraisemblance et le contrôle de la taille du pas. Nous commencerons par l'estimation du maximum de vraisemblance, qui consiste à accroître la probabilité d'obtenir des points performants. CMA-ES mettra à jour le centre de la population en calculant la moyenne pondérée des individus présentant une fitness élevée.

In [ ]:
X = np.arange(-5, 5, 0.1)
Y = np.arange(-5, 5, 0.1)
X, Y = np.meshgrid(X, Y)
Z = himmelblau(X, Y)

In [ ]:
lam = 20
x = np.random.rand(2) * 4 - 2
N = np.random.normal(size=(lam,2))
x_t = x + N
fits = himmelblau(x_t[:, 0], x_t[:, 1])

Tout d'abord, nous sélectionnons les $\mu$ meilleurs individus en fonction de leurs valeurs de fitness. Nous pondérons ensuite leur importance selon leur rang de fitness, à l'aide d'un vecteur de poids statique $w$. Ce vecteur $w$ constitue un hyperparamètre de l'algorithme CMA-ES, bien qu'une échelle de poids logarithmiques normalisés standard soit généralement utilisée. Il convient de noter que le poids ne correspond pas aux valeurs de fitness elles-mêmes, mais uniquement au rang. Il a été démontré que cette approche favorise la recherche dans des espaces de recherche mal conditionnés.

In [ ]:
mu = 10
sorted_ids = np.argsort(fits)
w = np.log(mu + 1/2) - np.log(np.arange(1, mu+1))
w /= np.sum(w)
weighted = N[sorted_ids[:mu]] * w.reshape(mu, 1)
s = np.mean(weighted, axis=0)
s

Ce vecteur de direction $s$ est similaire à l'estimation du gradient issue de l'exercice précédent. Toutefois, dans l'algorithme CMA-ES, seuls les $\mu$ meilleurs individus sont utilisés pour calculer $s$ ; de plus, au lieu d'utiliser directement les valeurs de fitness, $s$ est calculé à l'aide de pondérations fondées sur le rang de fitness. Nous pouvons comparer les deux vecteurs.

In [ ]:
A = (fits - np.mean(fits)) / np.std(fits)
G = -np.dot(A, N) / lam

In [ ]:
fig = plt.figure(figsize=(8, 6))
norm = colors.Normalize(vmin=Z.min(), vmax=Z.max())
cs = plt.contour(X, Y, Z, levels=100, cmap='Spectral', alpha=0.4, norm=norm)
plt.scatter(x_t[:, 0], x_t[:, 1], c=fits, cmap='Spectral', norm=norm)
plt.quiver(x[0], x[1], G[0], G[1], color='b', scale=4)
plt.quiver(x[0], x[1], s[0], s[1], color='g', scale=1)
plt.scatter(x[0], x[1], c='k')
fig.colorbar(cs)
plt.show()

Bien que nous n'utilisions les informations de fitness qu'à des fins de pondération par rang, ce vecteur de direction $s$ peut être considéré comme une approximation du gradient. Comme précédemment, nous pouvons exploiter cette information de gradient pour mettre à jour le centre de notre distribution. Il s'agit de la première étape de l'estimation du maximum de vraisemblance, consistant à aligner le centre de la distribution sur la moyenne pondérée des meilleures solutions.

In [ ]:
x = x + s

L'une des idées fondamentales de l'algorithme CMA-ES consiste à estimer la covariance entre les échantillons sélectionnés et à utiliser cette matrice de covariance pour façonner la distribution suivante. Une matrice de covariance mesure la variabilité conjointe entre les différentes dimensions de la fonction objectif. Les matrices de covariance sortent du cadre de ce cours, mais cet article fournit davantage de détails: https://datascienceplus.com/understanding-the-covariance-matrix/

Nous lançons CMA-ES avec la matrice identité comme matrice de covariance initiale, en supposant que chaque variable est indépendante et présente un écart-type de 1.

In [ ]:
C = np.eye(2)
np.outer(s, (s).T)

À chaque étape, nous mettons à jour $C = C + ss^T$. Cette opération utilise la moyenne pondérée des échantillons comme estimation de la covariance. Dans l'algorithme CMA-ES, on parle de mise à jour de la covariance de rang un. Pour les problèmes de dimension supérieure, une mise à jour matricielle supplémentaire est effectuée.

In [ ]:
C = C + np.outer(s, (s).T)
C

La décomposition en valeurs propres de la matrice de covariance est ensuite utilisée pour transformer la distribution normale afin de l'adapter à l'espace de recherche. Il s'agit de la seconde étape de l'estimation du maximum de vraisemblance : en tirant des échantillons d'une distribution ajustée aux zones performantes de l'espace de recherche, on augmente la probabilité d'obtenir de bons individus. La moyenne de la distribution, $x$, ainsi que la forme définie par $C$, maximisent conjointement cette vraisemblance.

In [ ]:
N = np.random.normal(size=(lam, 2))
x_N = x + N
eigenvalues, eigenvectors = np.linalg.eig(C)
x_C = x + np.array([np.dot(eigenvectors, np.sqrt(eigenvalues) * N[i, :]) for i in range(lam)])

In [ ]:
fig = plt.figure(figsize=(8, 6))
norm = colors.Normalize(vmin=Z.min(), vmax=Z.max())
cs = plt.contour(X, Y, Z, levels=100, cmap='Spectral', alpha=0.4, norm=norm)
plt.scatter(x_N[:, 0], x_N[:, 1], c='k', alpha=0.3)
plt.scatter(x_C[:, 0], x_C[:, 1], c='b')
plt.scatter(x[0], x[1], c='r')
fig.colorbar(cs)
plt.show()

Nous pouvons considérer cette amélioration isolément et l'ajouter à l'ES canonique défini précédemment. La seule modification consiste à transformer la distribution normale à l'aide de la matrice de covariance avant d'y effectuer l'échantillonnage.

In [ ]:
def covariance_es(x, fitness, gens=200, mu=10, lam=20, verbose=False):
    # best values and fitness vector for logging
    x_best = x
    f_best = fitness(x)
    fits = np.zeros(gens)
    # calculate weight list and first covariance matrix
    w = np.log(mu + 1/2) - np.log(np.arange(1, mu+1))
    w /= np.sum(w)
    C = np.eye(2)
    # generation loop
    for g in range(gens):
        # calculate new individuals
        N = np.random.normal(size=(lam,2))
        x_N = x + N
        eigenvalues, eigenvectors = np.linalg.eig(C)
        x_C = x + np.array([np.dot(eigenvectors, np.sqrt(eigenvalues) * N[i, :]) for i in range(lam)])
        # evaluate individuals
        F = [fitness(x_C[i, :]) for i in range(lam)]
        # store best individual for logging
        for i in range(lam):
            if F[i] < f_best:
                f_best = F[i]
                x_best = x_C[i]
                if verbose:
                    print(g, " ", f_best)
        fits[g] = f_best
        # use weighted sum to update center
        sorted_ids = np.argsort(F)
        weighted = N[sorted_ids[:mu]] * w.reshape(mu, 1)
        s = np.mean(weighted, axis=0)
        x = x + s
        C = C + np.outer(s, (s).T)
    return fits, x_best

Comparez ce nouvel ES à l'ES canonique. L'inclusion de la matrice de covariance améliore-t-elle les résultats ?

In [ ]:
compare_es([canonical, covariance_es], himmelblau_v)

# <a name="cmaes">5.</a> CMAES part 2: Step size adaptation

Pour comprendre la deuxième caractéristique principale de CMA-ES, à savoir l'adaptation de la taille du pas, revenons à la première stratégie d'évolution (ES) définie : la $(1+\lambda)$-ES. Nous pouvons intégrer un hyperparamètre important à cette méthode : la taille de la distribution normale, ou son écart-type.

In [ ]:
def oneplus_lambda(x, f, std=1, gens=200, lam=20):
    x_best = x
    f_best = f(x)
    fits = np.zeros(gens)
    for g in range(gens):
        N = np.random.normal(size=(lam, len(x))) * std
        for i in range(lam):
            ind = x + N[i, :]
            fi = f(ind)
            if fi < f_best:
                f_best = fi
                x_best = ind
        x = x_best
        fits[g] = f_best
    return fits, x_best

En comparant différentes valeurs d'écart-type, nous pouvons constater que cela a un impact majeur sur le résultat final.

In [ ]:
f0 = lambda x, f: oneplus_lambda(x, f, std=0.01)
f1 = lambda x, f: oneplus_lambda(x, f, std=0.1)
f2 = lambda x, f: oneplus_lambda(x, f, std=1)
f3 = lambda x, f: oneplus_lambda(x, f, std=10)

compare_es([f0, f1, f2, f3], himmelblau_v)

Outre l'estimation du maximum de vraisemblance, la seconde caractéristique majeure de l'algorithme CMA-ES est le contrôle de la taille du pas. Ce mécanisme s'apparente fortement au paramètre de température du recuit simulé : il évolue au fil du temps pour favoriser soit une plus grande exploration, soit une précision accrue.

Modifions la taille du pas au cours de l'algorithme d'évolution $(1+\lambda)$ appliqué à la fonction de Himmelblau.     

On peut réduire légèrement `std` à chaque étape, utiliser une décroissance exponentielle ou appliquer une règle similaire à celle du recuit simulé.  
  
Appliquons ensuite cette stratégie à la fonction de Rosenbrock. Cette approche surpasse-t-elle la version standard $(1+\lambda)$ ?

In [ ]:
def oneplus_lambda_adaptive(x, f, std0=1.0, decay=0.01, gens=200, lam=20):
    x_best = x.copy()
    f_best = f(x_best)
    fits = np.zeros(gens)

    for g in range(gens):
        std = std0 * np.exp(-decay * g)

        N = np.random.normal(size=(lam, len(x))) * std
        for i in range(lam):
            ind = x + N[i,:]
            fi = f(ind)
            if fi < f_best:
                f_best = fi
                x_best = ind
        x = x_best
        fits[g] = f_best
    return fits, x_best


In [ ]:
# Test sur himmelblau
compare_es([oneplus_lambda, oneplus_lambda_adaptive], himmelblau_v)

Le mécanisme de contrôle de la taille du pas utilisé dans CMA-ES s'appuie sur l'estimation du gradient pour déterminer si cette taille doit être augmentée (exploration) ou réduite (exploitation). [Cet article](http://www.cmap.polytechnique.fr/~nikolaus.hansen/es-overview-2015.pdf) présente une vue d'ensemble des différentes méthodes ES, y compris le contrôle de la taille du pas employé par CMA-ES. 

<img src="https://github.com/d9w/evolution/raw/master/imgs/cmaes_onefifth.png" width="700">

<img src="https://github.com/d9w/evolution/raw/master/imgs/cmaes_pathlength.png" width="700">

<img src="https://github.com/d9w/evolution/raw/master/imgs/cmaes_pathlength2.png" width="700">

Nous disposons désormais des deux composantes de l'algorithme CMA-ES : l'estimation du maximum de vraisemblance et le contrôle de la taille du pas. Ces deux éléments se conjuguent pour modifier la distribution — tant par sa forme que par sa taille — au cours de la recherche, comme l'illustre l'animation ci-dessous

<img src="https://user-images.githubusercontent.com/5564044/73486622-db5cff00-43e8-11ea-98fb-8246dbacab6d.gif">

<img src="https://github.com/SupaeroDataScience/stochastic/blob/master/notebooks/imgs/cmaes.png?raw=1">

Pour voir CMA-ES en action, nous utiliserons le package pycma (https://github.com/CMA-ES/pycma). Si vous ne l'avez pas encore installé, faites-le ici :

In [ ]:
!pip install cma

`pycma` est activement maintenue par l'auteur de CMA-ES et propose bien plus d'astuces algorithmiques et de fonctionnalités que celles que nous avons abordées ici. La référence [4] offre un bon aperçu des différentes variantes de CMA-ES.  

[4] Hansen, Nikolaus. "The CMA evolution strategy: a comparing review." Towards a new evolutionary computation. Springer, Berlin, Heidelberg, 2006. 75-102.

In [ ]:
import cma
es = cma.CMAEvolutionStrategy(2*[0], 0.1, {'popsize': 20, 'verb_disp': 1})

In [ ]:
solutions = np.array(es.ask())
es.tell(solutions, [himmelblau(x[0], x[1]) for x in solutions])
es.disp_annotation()
es.disp();

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = plt.subplot()
norm = colors.Normalize(vmin=Z.min(), vmax=Z.max())
cs = plt.contour(X, Y, Z, levels=100, cmap='Spectral', alpha=0.4, norm=norm)
plt.scatter(solutions[:, 0], solutions[:, 1])
fig.colorbar(cs)
plt.show()

In [ ]:
def generation(i, es, ax):
    solutions = np.array(es.ask())
    es.tell(solutions, [himmelblau(x[0], x[1]) for x in solutions])

    ax.clear()
    ax.contour(X, Y, Z, levels=100, cmap='Spectral', alpha=0.4, norm=norm)
    ax.scatter(solutions[:, 0], solutions[:, 1])
    ax.set_ylim(-5, 5)
    ax.set_xlim(-5, 5)
    ax.plot()

In [ ]:
es = cma.CMAEvolutionStrategy(2 * [0], 0.1, {'popsize': 20})

In [ ]:
import matplotlib.animation as animation
from IPython import display
fig = plt.figure(figsize=(8,6))
ax = plt.subplot()
fig.tight_layout()
frames = 20
animator = animation.FuncAnimation(fig, generation, fargs=(es, ax), frames=frames, interval=250, blit=False)
display.display(display.HTML(animator.to_html5_video()))
plt.close();

Les stratégies d'évolution (ES), et en particulier le CMA-ES, sont largement utilisées dans de nombreuses applications. Toute problématique pouvant être modélisée comme un problème d'optimisation continue peut faire appel à une stratégie d'évolution, et le CMA-ES représente l'état de l'art dans ce domaine. Voici une liste non exhaustive d'applications :  

+ Kämpf, J. H., & Robinson, D. (2009). A hybrid CMA-ES and HDE optimisation algorithm with application to solar energy potential. Applied Soft Computing, 9(2), 738-745.
+ Maki, A., Sakamoto, N., Akimoto, Y., Nishikawa, H., & Umeda, N. (2020). Application of optimal control theory based on the evolution strategy (CMA-ES) to automatic berthing. Journal of Marine Science and Technology, 25(1), 221-233.
+ Loshchilov, I., & Hutter, F. (2016). CMA-ES for hyperparameter optimization of deep neural networks. arXiv preprint arXiv:1604.07269.
+ Fukagata, K., Kern, S., Chatelain, P., Koumoutsakos, P., & Kasagi, N. (2008). Evolutionary optimization of an anisotropic compliant surface for turbulent friction drag reduction. Journal of Turbulence, (9), N35.

Pour finir, comparons l'algorithme CMA-ES aux variantes $(1+\lambda)$, $(\mu, \lambda)$ et canonique. 

In [ ]:
# On définit en premier lieu une fonction cma-es
def cma_es(x, f, gens=200, lam=20):
    import cma
    es = cma.CMAEvolutionStrategy(x, 0.1, {'popsize': 20})
    fits = np.zeros(gens)

    for g in range(gens):
        X = es.ask()
        F = [f(xi) for xi in X]
        es.tell(X, F)
        fits[g] = np.min(F)

    return fits, es.mean

In [ ]:
compare_es([oneplus_lambda, mu_lambda, canonical, cma_es], rosenbrock_v)